<a href="https://colab.research.google.com/github/Vidogreg/nlp-summer-school-2026/blob/main/demos/omnivoice/text_to_speech_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OmniVoice TTS demo (Colab)

Generates speech with [k2-fsa/OmniVoice](https://huggingface.co/k2-fsa/OmniVoice). Caches model weights to Google Drive so you don't re-download a few GB every time you open this notebook.

**Note:** if you share one Drive folder with many workshop attendees, Google Drive's anti-abuse download quota can start throttling it under heavy simultaneous use. This notebook falls back to a normal Hugging Face download if the Drive cache doesn't work.

In [1]:
# Colab already ships a working torch/CUDA pair, so only install what's missing.
!pip install -q omnivoice soundfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.5/168.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 4.7 MB/s eta 0:00:00


In [4]:
import os
from google.colab import drive

drive.mount("/content/drive")

# Point this at a folder in your Drive. For a shared workshop copy: share a
# folder from your own Drive, have attendees "Add shortcut to Drive", then
# set this to that shortcut's path in their Drive (e.g.
# "/content/drive/MyDrive/omnivoice-shared-cache").
DRIVE_CACHE_DIR = "/content/drive/MyDrive/omnivoice_cache"
os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
os.environ["HF_HOME"] = DRIVE_CACHE_DIR

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import torch
from omnivoice import OmniVoice

MODEL_ID = "k2-fsa/OmniVoice"
device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device.startswith("cuda") else torch.float32

try:
    model = OmniVoice.from_pretrained(MODEL_ID, device_map=device, dtype=dtype)
except Exception as e:
    print(f"Drive cache unavailable ({e!r}); falling back to a fresh download.")
    os.environ.pop("HF_HOME", None)
    model = OmniVoice.from_pretrained(MODEL_ID, device_map=device, dtype=dtype)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

In [12]:
import soundfile as sf
from IPython.display import Audio, display

# `instruct` must use only exact tokens from the model's fixed vocabulary, e.g.:
# male, female, child, elderly, teenager, young adult, middle-aged, low pitch,
# high pitch, moderate pitch, british accent, indian accent, ... (comma-separated,
# English or Chinese only, not mixed).
# `language` is optional: a code ("sk") or full name ("Slovak"), 600+ supported.
EXAMPLES = [
    {
        "name": "male_low_pitch",
        "text": "Hello NLP Summer School of 2026. Welcome in Kinit.",
        "instruct": "female",
    },
    {
        "name": "slovak_female",
        "text": "Ahoj, letná škola NLP 2026. Vitajte v Kinite...",
        "instruct": "male",
        "language": "sk",
    },
    {
        "name": "slovak_female",
        "text": "Ahoj, letná škola NLP 2026. Vitajte v Kinyte. ",
        "instruct": "male",
        "language": "sk",
    },
]

os.makedirs("outputs", exist_ok=True)

for example in EXAMPLES:
    print(f"Generating '{example['name']}'...")
    audio = model.generate(
        text=example["text"],
        instruct=example["instruct"],
        language=example.get("language"),
    )

    if isinstance(audio, list):
        audio = audio[0]

    output_path = f"outputs/{example['name']}.wav"
    sf.write(output_path, audio, 24000)
    display(Audio(output_path))

Generating 'male_low_pitch'...


Generating 'slovak_female'...


Generating 'slovak_female'...
